# Day 47 — Worked solutions

Run top to bottom in Colab. Share after individual work.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import expit, logsumexp

np.set_printoptions(precision=4, suppress=True)

def step(z):
    return (np.asarray(z) >= 0).astype(int)

def sigmoid(z):
    return expit(z)

def relu(z):
    return np.maximum(0, z)

def leaky_relu(z, alpha=.01):
    z = np.asarray(z)
    return np.where(z >= 0, z, alpha*z)

def softmax(logits):
    logits = np.asarray(logits, dtype=float)
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exps = np.exp(shifted)
    return exps / np.sum(exps, axis=-1, keepdims=True)

def bce_from_logits(z,y):
    return np.logaddexp(0,z) - np.asarray(y)*np.asarray(z)

def ce_from_logits(logits,true_index):
    logits=np.asarray(logits,dtype=float)
    return logsumexp(logits) - logits[true_index]

## Question 1 — Evaluate activations

Step [0,1,1]; sigmoid [.119,.500,.881]; tanh [-.964,0,.964]; ReLU [0,0,2]; Leaky [-.02,0,2]. Tanh and Leaky ReLU can be negative.

In [ ]:
z=np.array([-2.,0.,2.]);print(step(z),np.round(sigmoid(z),3),np.round(np.tanh(z),3),relu(z),leaky_relu(z))

## Question 2 — Reuse Day 46

z=.4, sigmoid≈.5987, class 1. This is a model probability estimate; sigmoid alone does not guarantee calibration.

In [ ]:
z=np.array([1.,1.])@np.array([.8,.6])-1;print(z,sigmoid(z),int(sigmoid(z)>=.5))

## Question 3 — Saturation

Sigmoid ≈ [.5000,.8808,.9997]; slopes ≈ [.2500,.1050,.000335]. The slope at 8 is smallest. Repeated small derivatives can weaken gradients.

In [ ]:
z=np.array([0.,2.,8.]);p=sigmoid(z);print(p,p*(1-p))

## Question 4 — Stable softmax

Probabilities [.6652,.2447,.0900] sum to 1. Adding a constant cancels in numerator and denominator. Subtracting the maximum keeps exponentials finite.

In [ ]:
a=softmax([2.,1.,0.]);print(a,a.sum(),softmax([1002.,1001.,1000.]))

## Question 5 — Choose outputs

Linear; sigmoid; softmax; one sigmoid per independent tag. Softmax forces a sum of 1 even if dog and tree are both present.

In [ ]:
print('linear, sigmoid, softmax, independent sigmoids')

## Question 6 — MSE and MAE

Errors y-ŷ are [1,-3]. MSE=(1+9)/2=5. MAE=(1+3)/2=2. MSE emphasizes the second miss.

In [ ]:
y=np.array([3.,7.]);p=np.array([2.,10.]);print(y-p,np.mean((y-p)**2),np.mean(abs(y-p)))

## Question 7 — BCE for a positive

For y=1, losses are -ln(.8)≈.2231 and -ln(.1)≈2.3026. A low assigned probability for the true class costs much more.

In [ ]:
print(-np.log(.8),-np.log(.1))

## Question 8 — BCE for a negative

For y=0,p=.8: -ln(.2)≈1.6094. For y=1,p=.2, the same -ln(.2) appears.

In [ ]:
print(-np.log(1-.8),-np.log(.2))

## Question 9 — Multiclass loss

Softmax [.6652,.2447,.0900]. True bird costs ≈2.4076; true cat costs ≈.4076. The losses differ by 2.

In [ ]:
z=np.array([2.,1.,0.]);print(softmax(z),ce_from_logits(z,2),ce_from_logits(z,0))

## Question 10 — Accuracy and loss

Both threshold to class 1. BCE values ≈.6733 and .01005. Accuracy ignores confidence once the hard label is correct.

In [ ]:
print(-np.log(.51),-np.log(.99))

## Question 11 — Numerical stability

Sigmoid(-1000) can underflow to zero, then log(0) is nonfinite. Stable BCE from logits is 1000.

In [ ]:
print(bce_from_logits(-1000.,1))

## Question 12 — Diagnose the call

Both routines expect raw logits. Use BCEWithLogitsLoss(z,y) and CrossEntropyLoss(logits,class_index). Apply sigmoid/softmax separately for display.

In [ ]:
print('binary: raw logit; multiclass: raw class logits')